In [0]:
%sql
-- 1. Criar catálogo
CREATE CATALOG IF NOT EXISTS proj5k_dw;
USE CATALOG proj5k_dw;

-- 2. Criar schemas
CREATE SCHEMA IF NOT EXISTS landing_manual;
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS dw_star;
CREATE SCHEMA IF NOT EXISTS dw_snowflake;


In [0]:
%sql
SELECT current_catalog(), current_schema();


In [0]:
%sql
-- Criando nossa tabela base, a partir dos dados da landing
CREATE OR REPLACE TABLE proj5k_dw.landing_manual.sales_orders_raw AS -- CTAs
SELECT *
FROM read_files(
  '/Volumes/proj5k_dw/landing_manual/input/sales_orders (1).csv',
  header => 'true',
  inferSchema => 'true'
)

In [0]:
%sql
SELECT *
FROM proj5k_dw.landing_manual.sales_orders_raw
LIMIT 5

In [0]:
%sql
USE SCHEMA bronze;

CREATE OR REPLACE TABLE sales_orders AS
SELECT
  CAST(order_id AS STRING) AS order_id,
  CAST(order_item_id AS INT) AS order_item_id,
  CAST(order_date AS DATE) AS order_date,
  CAST(shipment_date AS DATE) AS shipment_date,
  CAST(customer_id AS STRING) AS customer_id,
  customer_name,
  customer_email,
  CAST(customer_birthdate AS DATE) AS customer_birthdate,
  customer_gender,
  customer_city,
  customer_state,
  CAST(product_id AS STRING) AS product_id,
  product_name,
  product_category,
  product_subcategory,
  brand,
  CAST(unit_price AS DECIMAL(10,2)) AS unit_price,
  CAST(quantity AS INT) AS quantity,
  CAST(discount_value AS DECIMAL(10,2)) AS discount_value,
  CAST(store_id AS STRING) AS store_id,
  store_name,
  store_city,
  store_state,
  payment_method,
  CAST(payment_installments AS INT) AS payment_installments,
  payment_status,
  -- métricas calculadas
  (unit_price * quantity) AS total_gross,
  (unit_price * quantity) - discount_value AS total_net
FROM landing_manual.sales_orders_raw;


In [0]:
%sql
SELECT *
FROM proj5k_dw.bronze.sales_orders
LIMIT 10

### Desenvolvimento de dimensões e fato na modelagem estrela

In [0]:
%sql
USE SCHEMA dw_star;

CREATE OR REPLACE TABLE dim_date (
  date_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  date_value DATE,
  year INT,
  month INT,
  day INT,
  month_name STRING
);

INSERT INTO dim_date (date_value, year, month, day, month_name)
SELECT DISTINCT
  order_date AS date_value,
  YEAR(order_date),
  MONTH(order_date),
  DAY(order_date),
  DATE_FORMAT(order_date, 'MMMM')
FROM bronze.sales_orders
WHERE order_date IS NOT NULL;


In [0]:
%sql
SELECT *
FROM proj5k_dw.dw_star.dim_date
LIMIT 10

In [0]:
%sql
-- não usamos a dim_star, porque o use já foi carregado da celula anterior

CREATE OR REPLACE TABLE dim_customer (
  customer_sk BIGINT GENERATED ALWAYS AS IDENTITY, -- "ROW_NUMBER()" 
  customer_id STRING,
  customer_name STRING,
  customer_email STRING,
  customer_birthdate DATE,
  customer_gender STRING,
  customer_city STRING,
  customer_state STRING
);

INSERT INTO dim_customer (
  customer_id, customer_name, customer_email,
  customer_birthdate, customer_gender,
  customer_city, customer_state
)
SELECT DISTINCT
  customer_id, customer_name, customer_email,
  customer_birthdate, customer_gender,
  customer_city, customer_state
FROM bronze.sales_orders;


In [0]:
%sql
SELECT *
FROM proj5k_dw.dw_star.dim_customer
LIMIT 5


In [0]:
%sql
CREATE OR REPLACE TABLE dim_product (
  product_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  product_id STRING,
  product_name STRING,
  product_category STRING,
  product_subcategory STRING,
  brand STRING
);

INSERT INTO dim_product (
  product_id, product_name, product_category,
  product_subcategory, brand
)
SELECT DISTINCT
  product_id, product_name, product_category,
  product_subcategory, brand
FROM bronze.sales_orders;


In [0]:
%sql
SELECT *
FROM proj5k_dw.dw_star.dim_product
LIMIT 5

In [0]:
%sql
CREATE OR REPLACE TABLE dim_store (
  store_sk   BIGINT GENERATED ALWAYS AS IDENTITY,
  store_id   STRING,
  store_name STRING,
  store_city STRING,
  store_state STRING
);

INSERT INTO dim_store (
  store_id,
  store_name,
  store_city,
  store_state
)
SELECT DISTINCT
  store_id,
  store_name,
  store_city,
  store_state
FROM proj5k_dw.bronze.sales_orders;


In [0]:
%sql
SELECT *
FROM dim_store
LIMIT 5

In [0]:
%sql
USE CATALOG proj5k_dw;
USE SCHEMA dw_star;

CREATE OR REPLACE TABLE dim_payment (
  payment_sk          BIGINT GENERATED ALWAYS AS IDENTITY,
  payment_method      STRING,
  payment_installments INT,
  payment_status      STRING
);

INSERT INTO dim_payment (
  payment_method,
  payment_installments,
  payment_status
)
SELECT DISTINCT
  payment_method,
  payment_installments,
  payment_status
FROM proj5k_dw.bronze.sales_orders;


In [0]:
%sql
SELECT *
FROM proj5k_dw.dw_star.dim_payment
WHERE payment_method = 'Cartão de Crédito'
LIMIT 5

In [0]:
%sql
USE CATALOG proj5k_dw;
USE SCHEMA dw_star;

CREATE OR REPLACE TABLE fact_order_items AS -- CTAs (=/= CTE)
SELECT
  b.order_id,
  b.order_item_id,

  -- chaves substitutas / técnicas
  d_date.date_sk         AS order_date_sk,
  d_cust.customer_sk     AS customer_sk,
  d_prod.product_sk      AS product_sk,
  d_store.store_sk       AS store_sk,
  d_pay.payment_sk       AS payment_sk,

  -- métricas numéricas
  b.quantity,
  b.unit_price,
  b.discount_value,
  b.total_gross,
  b.total_net

FROM proj5k_dw.bronze.sales_orders b
JOIN dw_star.dim_date      d_date
  ON b.order_date = d_date.date_value
JOIN dw_star.dim_customer  d_cust
  ON b.customer_id = d_cust.customer_id
JOIN dw_star.dim_product   d_prod
  ON b.product_id = d_prod.product_id
JOIN dw_star.dim_store     d_store
  ON b.store_id = d_store.store_id
JOIN dw_star.dim_payment   d_pay
  ON  b.payment_method      = d_pay.payment_method
  AND b.payment_installments = d_pay.payment_installments
  AND b.payment_status      = d_pay.payment_status;


In [0]:
%sql
SELECT *
FROM proj5k_dw.dw_star.fact_order_items
LIMIT 5

~~~sql
Table dim_date {
  date_sk      bigint [pk, note: 'Surrogate key']
  date_value   date   [not null]
  year         int
  month        int
  day          int
  month_name   varchar
}

Table dim_customer {
  customer_sk        bigint [pk, note: 'Surrogate key']
  customer_id        varchar [not null]
  customer_name      varchar
  customer_email     varchar
  customer_birthdate date
  customer_gender    varchar
  customer_city      varchar
  customer_state     varchar
}

Table dim_product {
  product_sk         bigint [pk, note: 'Surrogate key']
  product_id         varchar [not null]
  product_name       varchar
  product_category   varchar
  product_subcategory varchar
  brand              varchar
}

Table dim_store {
  store_sk    bigint [pk, note: 'Surrogate key']
  store_id    varchar [not null]
  store_name  varchar
  store_city  varchar
  store_state varchar
}

Table dim_payment {
  payment_sk          bigint [pk, note: 'Surrogate key']
  payment_method      varchar
  payment_installments int
  payment_status      varchar
}

Table fact_order_items {
  order_id        varchar
  order_item_id   int

  order_date_sk   bigint [not null]
  customer_sk     bigint [not null]
  product_sk      bigint [not null]
  store_sk        bigint [not null]
  payment_sk      bigint [not null]

  quantity        int
  unit_price      decimal(10,2)
  discount_value  decimal(10,2)
  total_gross     decimal(10,2)
  total_net       decimal(10,2)

  Indexes {
    (order_id, order_item_id) [name: 'idx_fact_order_pk_like']
  }
}

Ref: fact_order_items.order_date_sk > dim_date.date_sk
Ref: fact_order_items.customer_sk  > dim_customer.customer_sk
Ref: fact_order_items.product_sk   > dim_product.product_sk
Ref: fact_order_items.store_sk     > dim_store.store_sk
Ref: fact_order_items.payment_sk   > dim_payment.payment_sk
~~~
Que nos gera um belíssimo diagrama:

![Diagrama Star Schema](/Workspace/Users/marcocaja.dataengineer@gmail.com/Aulas - Projeto 5K/Ao Vivo - 20251124/files/Diagrama Star Schema.png)

In [0]:
%sql
-- Gerar nossa tabela agregada analisando o desempenho de categorias de produtos, por Estado e ano
SELECT
  d_date.year,
  d_prod.product_category,
  d_store.store_state,
  SUM(f.total_net) AS revenue
FROM dw_star.fact_order_items f
JOIN dw_star.dim_date   d_date   ON f.order_date_sk = d_date.date_sk
JOIN dw_star.dim_product d_prod  ON f.product_sk    = d_prod.product_sk
JOIN dw_star.dim_store  d_store  ON f.store_sk      = d_store.store_sk
GROUP BY
  d_date.year,
  d_prod.product_category,
  d_store.store_state
ORDER BY
  d_date.year,
  d_prod.product_category,
  d_store.store_state;


### Seguindo para um Snowflake Schema

In [0]:
%sql
USE SCHEMA dw_snowflake;

CREATE OR REPLACE TABLE dim_category (
  category_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  product_category STRING
);

INSERT INTO dim_category (product_category)
SELECT DISTINCT product_category
FROM bronze.sales_orders;


In [0]:
%sql
SELECT *
FROM proj5k_dw.dw_snowflake.dim_category
LIMIT 5

In [0]:
%sql
CREATE OR REPLACE TABLE dim_subcategory (
  subcategory_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  product_subcategory STRING,
  category_sk BIGINT
);

INSERT INTO dim_subcategory (product_subcategory, category_sk)
SELECT DISTINCT
  b.product_subcategory,
  c.category_sk
FROM bronze.sales_orders b
JOIN dw_snowflake.dim_category c
  ON b.product_category = c.product_category;


In [0]:
%sql
SELECT *
FROM proj5k_dw.dw_snowflake.dim_subcategory
LIMIT 5


In [0]:
%sql
CREATE OR REPLACE TABLE dim_brand (
  brand_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  brand STRING
);

INSERT INTO dim_brand (brand)
SELECT DISTINCT brand
FROM bronze.sales_orders;


In [0]:
%sql
SELECT *
FROM proj5k_dw.dw_snowflake.dim_brand
LIMIT 5

In [0]:
%sql
-- Aqui nós estamos na dw_snowflake e não na dw_star. Se quiser, pode usar o nome qualificado
-- com o three level namespace, para facilitar a visualização.
CREATE OR REPLACE TABLE dim_product (
  product_sk BIGINT GENERATED ALWAYS AS IDENTITY,
  product_id STRING,
  product_name STRING,
  subcategory_sk BIGINT,
  brand_sk BIGINT
);

INSERT INTO dim_product (product_id, product_name, subcategory_sk, brand_sk)
SELECT DISTINCT
  b.product_id,
  b.product_name,
  s.subcategory_sk,
  br.brand_sk
FROM bronze.sales_orders b
JOIN dw_snowflake.dim_subcategory s
  ON b.product_subcategory = s.product_subcategory
JOIN dw_snowflake.dim_brand br
  ON b.brand = br.brand;


In [0]:
%sql
-- qualificada no schema dw_snowflake
SELECT *
FROM proj5k_dw.dw_snowflake.dim_product
LIMIT 5

In [0]:
%sql
-- qualificada no schema dw_star
SELECT * 
FROM proj5k_dw.dw_star.dim_product
LIMIT 5

In [0]:
%sql
CREATE OR REPLACE TABLE dim_state (
  state_sk   BIGINT GENERATED ALWAYS AS IDENTITY,
  state_code STRING
);

INSERT INTO dim_state (state_code)
SELECT DISTINCT
  store_state
FROM proj5k_dw.bronze.sales_orders
WHERE store_state IS NOT NULL;


In [0]:
%sql
SELECT *
FROM proj5k_dw.dw_snowflake.dim_state

In [0]:
%sql
CREATE OR REPLACE TABLE dim_city (
  city_sk   BIGINT GENERATED ALWAYS AS IDENTITY,
  city_name STRING,
  state_sk  BIGINT
);

INSERT INTO dim_city (city_name, state_sk)
SELECT DISTINCT
  b.store_city      AS city_name,
  s.state_sk        AS state_sk
FROM proj5k_dw.bronze.sales_orders b
JOIN dw_snowflake.dim_state s
  ON b.store_state = s.state_code
WHERE b.store_city IS NOT NULL;


In [0]:
%sql
SELECT *
FROM proj5k_dw.dw_snowflake.dim_city
LIMIT 5

In [0]:
%sql
CREATE OR REPLACE TABLE dim_store (
  store_sk   BIGINT GENERATED ALWAYS AS IDENTITY,
  store_id   STRING,
  store_name STRING,
  city_sk    BIGINT
);

INSERT INTO dim_store (store_id, store_name, city_sk)
SELECT DISTINCT
  b.store_id,
  b.store_name,
  c.city_sk
FROM proj5k_dw.bronze.sales_orders b
JOIN dw_snowflake.dim_city c
  ON  b.store_city  = c.city_name
JOIN dw_snowflake.dim_state s
  ON  b.store_state = s.state_code
  AND c.state_sk    = s.state_sk;


In [0]:
%sql
SELECT *
FROM proj5k_dw.dw_snowflake.dim_store
LIMIT 5

In [0]:
%sql
CREATE OR REPLACE TABLE proj5k_dw.dw_snowflake.fact_order_items AS
SELECT
  b.order_id,
  b.order_item_id,

  -- chaves técnicas
  d_date.date_sk        AS order_date_sk,
  d_cust.customer_sk    AS customer_sk,
  d_prod.product_sk     AS product_sk,
  d_store.store_sk      AS store_sk,
  d_pay.payment_sk      AS payment_sk,

  -- métricas
  b.quantity,
  b.unit_price,
  b.discount_value,
  b.total_gross,
  b.total_net

FROM proj5k_dw.bronze.sales_orders b
JOIN proj5k_dw.dw_star.dim_date      d_date
  ON b.order_date   = d_date.date_value
JOIN proj5k_dw.dw_star.dim_customer  d_cust
  ON b.customer_id  = d_cust.customer_id
JOIN proj5k_dw.dw_snowflake.dim_store d_store
  ON b.store_id     = d_store.store_id
JOIN proj5k_dw.dw_star.dim_payment   d_pay
  ON  b.payment_method      = d_pay.payment_method
  AND b.payment_installments = d_pay.payment_installments
  AND b.payment_status      = d_pay.payment_status
JOIN proj5k_dw.dw_snowflake.dim_product d_prod
  ON b.product_id  = d_prod.product_id;


In [0]:
%sql
SELECT *
FROM proj5k_dw.dw_snowflake.fact_order_items
LIMIT 5

In [0]:
%sql
SELECT *
FROM proj5k_dw.dw_star.fact_order_items
LIMIT 5

~~~sql
////////////////////////////////////////////////////
// Dimensões comuns (compartilhadas com o star)
////////////////////////////////////////////////////

Table dim_date {
  date_sk     bigint [pk, note: 'Surrogate key']
  date_value  date   [not null]
  year        int
  month       int
  day         int
  month_name  varchar
}

Table dim_customer {
  customer_sk        bigint [pk, note: 'Surrogate key']
  customer_id        varchar [not null]
  customer_name      varchar
  customer_email     varchar
  customer_birthdate date
  customer_gender    varchar
  customer_city      varchar
  customer_state     varchar
}

Table dim_payment {
  payment_sk           bigint [pk, note: 'Surrogate key']
  payment_method       varchar
  payment_installments int
  payment_status       varchar
}

////////////////////////////////////////////////////
// Snowflake de Produto
////////////////////////////////////////////////////

Table dim_category {
  category_sk      bigint [pk, note: 'Surrogate key']
  product_category varchar
}

Table dim_subcategory {
  subcategory_sk     bigint [pk, note: 'Surrogate key']
  product_subcategory varchar
  category_sk        bigint [not null]
}

Table dim_brand {
  brand_sk bigint [pk, note: 'Surrogate key']
  brand    varchar
}

Table dim_product {
  product_sk    bigint [pk, note: 'Surrogate key']
  product_id    varchar [not null]
  product_name  varchar
  subcategory_sk bigint [not null]
  brand_sk      bigint [not null]
}

////////////////////////////////////////////////////
// Snowflake de Geografia / Loja
////////////////////////////////////////////////////

Table dim_state {
  state_sk   bigint [pk, note: 'Surrogate key']
  state_code varchar
}

Table dim_city {
  city_sk   bigint [pk, note: 'Surrogate key']
  city_name varchar
  state_sk  bigint [not null]
}

Table dim_store {
  store_sk   bigint [pk, note: 'Surrogate key']
  store_id   varchar [not null]
  store_name varchar
  city_sk    bigint [not null]
}

////////////////////////////////////////////////////
// Fato (snowflake) – fact_order_items
////////////////////////////////////////////////////

Table fact_order_items {
  // Grão: 1 linha por (order_id, order_item_id)
  order_id      varchar
  order_item_id int

  order_date_sk bigint [not null]
  customer_sk   bigint [not null]
  product_sk    bigint [not null]
  store_sk      bigint [not null]
  payment_sk    bigint [not null]

  quantity      int
  unit_price    decimal(10,2)
  discount_value decimal(10,2)
  total_gross   decimal(10,2)
  total_net     decimal(10,2)

  Indexes {
    (order_id, order_item_id) [name: 'idx_fact_order_pk_like']
  }
}

////////////////////////////////////////////////////
// Relações
////////////////////////////////////////////////////

// Fato -> dimensões comuns
Ref: fact_order_items.order_date_sk > dim_date.date_sk
Ref: fact_order_items.customer_sk  > dim_customer.customer_sk
Ref: fact_order_items.payment_sk   > dim_payment.payment_sk

// Fato -> produto (snowflake)
Ref: fact_order_items.product_sk   > dim_product.product_sk

// Fato -> loja (snowflake de geografia)
Ref: fact_order_items.store_sk     > dim_store.store_sk

// Hierarquia de produto
Ref: dim_subcategory.category_sk   > dim_category.category_sk
Ref: dim_product.subcategory_sk    > dim_subcategory.subcategory_sk
Ref: dim_product.brand_sk          > dim_brand.brand_sk

// Hierarquia de geografia
Ref: dim_city.state_sk             > dim_state.state_sk
Ref: dim_store.city_sk             > dim_city.city_sk

~~~
Gerando um diagrama muito mais complexo:

![Diagram Snowflake Schema](/Workspace/Users/marcocaja.dataengineer@gmail.com/Aulas - Projeto 5K/Ao Vivo - 20251124/files/Diagram Snowflake Schema.png)

### Conclusão
Em uma aula direta, você:
- revisou o que é um data warehouse
- entendeu mais sobre as modelagens estrela (star schema) e floco de neve (snowflake schema)
- construiu um data warehouse completo, com dimensões e fatos, a partir de dados transacionais
- tem um projeto pronto para publicar no LinkedIn

In [0]:
%sql

DROP SCHEMA IF EXISTS proj5k_dw.dw_snowflake CASCADE;
DROP SCHEMA IF EXISTS proj5k_dw.dw_star      CASCADE;
DROP SCHEMA IF EXISTS proj5k_dw.bronze       CASCADE;
DROP SCHEMA IF EXISTS proj5k_dw.landing_manual CASCADE;

DROP CATALOG IF EXISTS proj5k_dw CASCADE;